In [1]:
import time

# departmanlar ve sunucular sabit, elle tanımladım
DEPARTMANLAR = ["AR_GE","Finans","Hukuk","IK","Satis","Yazilim","Yonetim"]
SUNUCULAR    = ["Arastirma_Srv","Bordro_Srv","Hukuk_Srv","Kod_Srv",
                "Musteri_Srv","Odeme_Srv","Proje_Srv","Yonetim_Srv"]

IZIN_SIRASI = {"No-Access":0,"Read":1,"Write":2,"Execute":3}
def sev(i): return IZIN_SIRASI[i]

# iki farklı değer sıralaması — tek fark bu
GREEDY = ["Execute","Write","Read","No-Access"]  # yüksekten düşüğe
LP     = ["No-Access","Read","Write","Execute"]  # least privilege

# iş gereksinimleri — her departmanın minimum erişim ihtiyacı
IS_GER = {
    ("AR_GE","Arastirma_Srv"): "Execute",
    ("AR_GE","Kod_Srv"):       "Read",
    ("AR_GE","Proje_Srv"):     "Write",
    ("Satis","Musteri_Srv"):   "Write",
    ("Satis","Kod_Srv"):       "Read",
    ("IK","Bordro_Srv"):       "Write",
    ("IK","Musteri_Srv"):      "Read",
    ("Finans","Odeme_Srv"):    "Write",
    ("Finans","Bordro_Srv"):   "Read",
    ("Finans","Musteri_Srv"):  "Read",
    ("Finans","Proje_Srv"):    "Read",
    ("Yazilim","Kod_Srv"):        "Execute",
    ("Yazilim","Proje_Srv"):      "Execute",
    ("Yazilim","Arastirma_Srv"):  "Read",
    ("Hukuk","Hukuk_Srv"):    "Execute",
    ("Hukuk","Musteri_Srv"):  "Read",
    ("Yonetim","Yonetim_Srv"):    "Execute",
    ("Yonetim","Bordro_Srv"):     "Read",
    ("Yonetim","Odeme_Srv"):      "Read",
    ("Yonetim","Musteri_Srv"):    "Read",
    ("Yonetim","Arastirma_Srv"):  "Read",
}

# kısıt fonksiyonu - kısmi atamayı da kontrol ediyor
def kisit(atama):
    def al(d,s): return atama.get((d,s))

    # iş gereksinimi karşılanıyor mu
    for (d,s),mn in IS_GER.items():
        v = al(d,s)
        if v is not None and sev(v) < sev(mn):
            return False, f"[IS] {d}->{s}: min '{mn}'"

    # K1-K6: departman bazlı yasak erişimler
    for srv in ["Bordro_Srv","Odeme_Srv","Hukuk_Srv","Yonetim_Srv"]:
        v = al("AR_GE",srv)
        if v and v!="No-Access": return False,f"[K1] AR_GE->{srv} yasak"

    for srv in ["Bordro_Srv","Odeme_Srv","Hukuk_Srv","Yonetim_Srv","Arastirma_Srv"]:
        v = al("Satis",srv)
        if v and v!="No-Access": return False,f"[K2] Satis->{srv} yasak"

    for srv in ["Kod_Srv","Odeme_Srv","Hukuk_Srv","Yonetim_Srv","Arastirma_Srv","Proje_Srv"]:
        v = al("IK",srv)
        if v and v!="No-Access": return False,f"[K3] IK->{srv} yasak"

    for srv in ["Kod_Srv","Hukuk_Srv","Yonetim_Srv","Arastirma_Srv"]:
        v = al("Finans",srv)
        if v and v!="No-Access": return False,f"[K4] Finans->{srv} yasak"

    for srv in ["Kod_Srv","Bordro_Srv","Odeme_Srv","Yonetim_Srv","Arastirma_Srv","Proje_Srv"]:
        v = al("Hukuk",srv)
        if v and v!="No-Access": return False,f"[K5] Hukuk->{srv} yasak"

    v = al("Yonetim","Kod_Srv")
    if v and v!="No-Access": return False,"[K6] Yonetim->Kod yasak"

    # K7-K9: görev ayrımı (SoD) kısıtları
    v = al("Finans","Odeme_Srv")
    if v=="Execute": return False,"[K7 SoD] Finans->Odeme Execute yasak"

    vb = al("Yonetim","Bordro_Srv"); vo = al("Yonetim","Odeme_Srv")
    if vb and vo and sev(vb)>=sev("Write") and sev(vo)>=sev("Execute"):
        return False,"[K8 SoD] Yonetim: Bordro-Write+Odeme-Execute yasak"

    vb2 = al("IK","Bordro_Srv"); vm = al("IK","Musteri_Srv")
    if vb2 and vm and vb2=="Write" and sev(vm)>=sev("Write"):
        return False,"[K9 SoD] IK: Bordro-Write+Musteri-Write yasak"

    # K10-K12: zero trust - hassas sunuculara kısıtlı erişim
    for d in DEPARTMANLAR:
        if d=="Yonetim": continue
        v = al(d,"Yonetim_Srv")
        if v and v!="No-Access": return False,f"[K10] {d}->Yonetim_Srv yasak"

    for d in DEPARTMANLAR:
        if d in ["Hukuk","Yonetim"]: continue
        v = al(d,"Hukuk_Srv")
        if v and v!="No-Access": return False,f"[K11] {d}->Hukuk_Srv yasak"

    for d in DEPARTMANLAR:
        if d in ["AR_GE","Yazilim","Yonetim"]: continue
        v = al(d,"Arastirma_Srv")
        if v and v!="No-Access": return False,f"[K12] {d}->Arastirma_Srv yasak"

    # K13: bir sunucuda en fazla 1 Execute olabilir
    for srv in SUNUCULAR:
        n = sum(1 for d in DEPARTMANLAR if atama.get((d,srv))=="Execute")
        if n>1: return False,f"[K13] {srv}: birden fazla Execute yetkisi"

    return True,"OK"


def mrv_sec(atanmamis, atama):
    # kaç geçerli değer kaldığını say, en azı seç
    def sayi(deg):
        n = 0
        for izin in ["No-Access","Read","Write","Execute"]:
            t = dict(atama); t[deg] = izin
            if kisit(t)[0]: n += 1
        return n
    return sorted(atanmamis, key=sayi)[0]


geri = 0
deneme = 0

def coz(atama, tum, use_mrv, degerler):
    global geri, deneme
    if len(atama)==len(tum): return dict(atama)
    atanmamis = [v for v in tum if v not in atama]
    sec = mrv_sec(atanmamis,atama) if use_mrv else atanmamis[0]
    for izin in degerler:
        deneme += 1
        atama[sec] = izin
        if kisit(atama)[0]:
            r = coz(atama,tum,use_mrv,degerler)
            if r: return r
        del atama[sec]; geri += 1
    return None


R={
    "No-Access":"\033[91m","Read":"\033[93m","Write":"\033[94m","Execute":"\033[92m",
    "baslik":"\033[1;30m","ihlal":"\033[1;91m","basari":"\033[1;92m",
    "gri":"\033[90m","sari":"\033[33m","_":"\033[0m"
}
def rc(m,t): return f"{R.get(t,'')}{m}{R['_']}"
def bslk(m): print("\n"+"="*70+"\n"+rc(f"  {m}","baslik")+"\n"+"="*70)

def matris(cozum):
    SEM={"No-Access":"X","Read":"R","Write":"W","Execute":"E"}
    KOL=16
    print()
    satir=f"{''*16:>16}"
    for s in SUNUCULAR:
        etiket = s.replace("_Srv","")
        satir+=f"{etiket:^{KOL}}"
    print(satir)
    print("  "+"-"*(14+KOL*len(SUNUCULAR)))
    for d in DEPARTMANLAR:
        satir=f"  {d:<16}"
        for s in SUNUCULAR:
            izin=cozum.get((d,s),"No-Access")
            hucre=f"{SEM[izin]}:{izin}"
            satir+=rc(f"{hucre:^{KOL}}",izin)
        print(satir)
    print()


def tablo3_yazdir(d1, g1, s1, d2, g2, s2):
    """performans sonuçlarını tablo formatında yazdırır"""
    print("\n" + "="*70)
    print(rc("  TABLO 3 — Algoritma Performans Karşılaştırması (Bildiri İçin)", "baslik"))
    print("="*70)
    print()
    satir_fmt = "  {:<35} {:>15} {:>15}"
    print(satir_fmt.format("Ölçüt", "Düz Backtracking", "MRV Backtracking"))
    print("  " + "-"*65)
    print(satir_fmt.format("Değişken sayısı",         "56",           "56"))
    print(satir_fmt.format("Değer sırası",  "Execute önce", "No-Access önce"))
    print(satir_fmt.format("Toplam deneme sayısı",    f"{d1:,}",       f"{d2:,}"))
    print(satir_fmt.format("Geri adım sayısı",        f"{g1:,}",       f"{g2:,}"))
    print(satir_fmt.format("Çalışma süresi (ms)",     f"~{s1*1000:.0f}", f"~{s2*1000:.0f}"))
    oran = round(d1/d2, 0) if d2 > 0 else "∞"
    print(satir_fmt.format("Deneme azalma oranı",     "—",            f"~{oran:,.0f}×"))
    print()
    print(rc("  → Bildirinizdeki Tablo 3'e bu değerleri aktarabilirsiniz.", "gri"))
    print("="*70 + "\n")


def main():
    global geri, deneme

    tum = [(d,s) for d in DEPARTMANLAR for s in SUNUCULAR]
    N   = len(tum)
    kaba = 4**N

    print(rc("""
+====================================================================+
|  CSP Tabanli Zero Trust Erisim Kontrol — MRV Fark Gosterimi       |
|  AKU 5. Muhendislik ve Ogrenci Sempozyumu                          |
+====================================================================+""","baslik"))

    print(f"\n  Departman     : {len(DEPARTMANLAR)}")
    print(f"  Sunucu        : {len(SUNUCULAR)}")
    print(f"  Degisken (|X|): {N}  ({len(DEPARTMANLAR)} x {len(SUNUCULAR)})")
    print(f"  Kaba kuvvet   : 4^{N} = {kaba:,} olasilik")
    print(f"  Kisit kurali  : 13  (K1–K13)")
    print(f"  Is gereksinimi: {len(IS_GER)}")

    print(rc("""
  ─────────────────────────────────────────────────────────────────
  ALGORITMA FARKLARI:
    Duz BT  → Degisken: alfabetik  | Deger: Execute→Write→Read→No-Access
    MRV BT  → Degisken: MRV        | Deger: No-Access→Read→Write→Execute
  ─────────────────────────────────────────────────────────────────""","gri"))

    bslk("SENARYO 1 — Duz Backtracking  (Greedy Deger Sirasi, Alfabetik Degisken)")
    geri=deneme=0
    t0=time.perf_counter(); c1=coz({},tum,False,GREEDY); s1=time.perf_counter()-t0
    print(f"  {rc('Cozum bulundu!','basari') if c1 else rc('Cozum bulunamadi!','ihlal')}")
    d1,g1=deneme,geri
    print(f"  Toplam deneme    : {rc(str(d1),  'ihlal'):>20}")
    print(f"  Geri adim sayisi : {rc(str(g1),  'ihlal'):>20}")
    print(f"  Sure             : {s1*1000:>10.3f} ms")

    bslk("SENARYO 2 — MRV Backtracking  (Least Privilege + MRV Degisken Secimi)")
    geri=deneme=0
    t0=time.perf_counter(); c2=coz({},tum,True,LP); s2=time.perf_counter()-t0
    print(f"  {rc('Cozum bulundu!','basari') if c2 else rc('Cozum bulunamadi!','ihlal')}")
    d2,g2=deneme,geri
    print(f"  Toplam deneme    : {rc(str(d2),  'basari'):>20}")
    print(f"  Geri adim sayisi : {rc(str(g2),  'basari'):>20}")
    print(f"  Sure             : {s2*1000:>10.3f} ms")

    bslk("PERFORMANS KARSILASTIRMASI")
    print(f"\n  {'Metrik':<35} {'Duz BT':>14} {'MRV BT':>14}  {'Tasarruf':>10}")
    print("  "+"-"*75)
    tden=d1-d2; tger=g1-g2
    print(f"  {'Toplam deneme':<35} {rc(str(d1),'ihlal'):>24} {rc(str(d2),'basari'):>24}  {rc(f'-{tden}','basari'):>20}")
    print(f"  {'Geri adim':<35} {rc(str(g1),'ihlal'):>24} {rc(str(g2),'basari'):>24}  {rc(f'-{tger}','basari'):>20}")
    if d2>0:
        oran=round(d1/d2,1)
        print(f"  {'Deneme azalmasi (oran)':<35} {'─':>14} {'─':>14}  {rc(f'{oran}x daha az','basari'):>20}")

    if c1 and c2:
        tablo3_yazdir(d1,g1,s1,d2,g2,s2)

    print(f"""
  {rc('NEDEN FARK OLUSTU?','sari')}
  ─────────────────────────────────────────────────────────────────
  K13 kisiti: Her sunucuda en fazla 1 Execute yetkisi.

  Duz BT (Execute-once dener):
    → AR_GE, Satis, Finans gibi departmanlar Kod_Srv ve Proje_Srv'da
      gereksiz yere Execute almaya calisir.
    → Yazilim bu sunucularda Execute alamayinca geri adim zorunlu olur.
    → Her catisma noktasi icin tekrarli geri adim + yeniden atama.

  MRV BT (Least Privilege + MRV):
    → (Yazilim, Kod_Srv) ve (Yazilim, Proje_Srv) icin tek gecerli deger
      Execute'tur (IS_GER gereksinimi). MRV bunlari ONCE cozumler.
    → Diger departmanlar K13 catismasi yasamadan No-Access/Read/Write alir.
    → Geri adim neredeyse yasanmaz.
  ─────────────────────────────────────────────────────────────────""")

    if c2:
        bslk("CSP COZUMU — ERISIM KONTROL MATRISI (MRV + Least Privilege)")
        matris(c2)
        print("  X=No-Access  R=Read  W=Write  E=Execute")
        print("  Renkler: ",end="")
        for iz in ["No-Access","Read","Write","Execute"]:
            print(rc(f" {iz} ",iz),end=" ")
        print()

    if c2:
        bslk("DETAYLI YETKI RAPORU")
        SEM={"No-Access":"X","Read":"R","Write":"W","Execute":"E"}
        for d in DEPARTMANLAR:
            print(f"\n  [{d}]")
            for s in SUNUCULAR:
                iz=c2.get((d,s),"No-Access")
                zor=" <- is gereksinimi" if (d,s) in IS_GER else ""
                conflict=" [K13 KRITIK]" if iz=="Execute" else ""
                print(f"    {SEM[iz]}  {s:<22} {rc(iz,iz)}{rc(zor,'gri')}{rc(conflict,'sari')}")

    print("\n"+"="*70+"\n")

if __name__=="__main__":
    main()



+====================================================================+
|  CSP Tabanli Zero Trust Erisim Kontrol — MRV Fark Gosterimi       |
|  AKU 5. Muhendislik ve Ogrenci Sempozyumu                          |
+====================================================================+

  Departman     : 7
  Sunucu        : 8
  Degisken (|X|): 56  (7 x 8)
  Kaba kuvvet   : 4^56 = 5,192,296,858,534,827,628,530,496,329,220,096 olasilik
  Kisit kurali  : 13  (K1–K13)
  Is gereksinimi: 21

  ─────────────────────────────────────────────────────────────────
  ALGORITMA FARKLARI:
    Duz BT  → Degisken: alfabetik  | Deger: Execute→Write→Read→No-Access
    MRV BT  → Degisken: MRV        | Deger: No-Access→Read→Write→Execute
  ─────────────────────────────────────────────────────────────────

  SENARYO 1 — Duz Backtracking  (Greedy Deger Sirasi, Alfabetik Degisken)
  Cozum bulundu!
  Toplam deneme    :    978305
  Geri adim sayisi :    978249
  Sure             :  22691.308 ms

  SENARYO 2 — MRV 